In [62]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

In [63]:
gold = pl.read_parquet("../data/raw/Gold_1d.parquet")
nifty = pl.read_parquet("../data/raw/Nifty50_1d.parquet")
usdinr = pl.read_parquet("../data/raw/USDINR_1d.parquet")

In [64]:
nifty = nifty.rename({"('Close', '^NSEI')":"Close","('High', '^NSEI')":"High","('Low', '^NSEI')":"Low","('Open', '^NSEI')":"Open","('Volume', '^NSEI')":"Volume"})
gold = gold.rename({"('Close', 'GOLDBEES.NS')":"Close","('High', 'GOLDBEES.NS')":"High","('Low', 'GOLDBEES.NS')":"Low","('Open', 'GOLDBEES.NS')":"Open","('Volume', 'GOLDBEES.NS')":"Volume"})
usdinr = usdinr.rename({"('Close', 'USDINR=X')":"Close","('High', 'USDINR=X')":"High","('Low', 'USDINR=X')":"Low","('Open', 'USDINR=X')":"Open","('Volume', 'USDINR=X')":"Volume"})


In [65]:
bad_dates = [
    pl.datetime(2019, 12, 19),
    pl.datetime(2019, 12, 20),
]
for date in bad_dates:
    nifty = nifty.filter(pl.col("Date") != date)
    gold = gold.filter(pl.col("Date") != date)
    usdinr = usdinr.filter(pl.col("Date") != date)


In [66]:
nifty = nifty.with_columns(
    pl.col("Close").pct_change().alias("Ret_1d"),
    pl.col("Close").pct_change(n=3).alias("Ret_3d"),
    pl.col("Close").pct_change(n=5).alias("Ret_5d"),
    pl.col("Close").pct_change(n=20).alias("Ret_20d"),
    pl.col("Close").rolling_mean(window_size=5).alias("MA_5d"),
    pl.col("Close").rolling_mean(window_size=20).alias("MA_20d"),
)
nifty = nifty.with_columns(
    pl.col("Ret_1d").rolling_std(window_size=5).alias("Vol_5d"),
    pl.col("Ret_1d").rolling_std(window_size=10).alias("Vol_10d"),
    pl.col("Ret_1d").rolling_std(window_size=20).alias("Vol_20d"),
)
nifty = nifty.with_columns(
    pl.col("Ret_1d").abs().alias("Abs_Return"),
    pl.col("Ret_1d").abs().rolling_mean(window_size=5).alias("Rolling_Abs_Return"),
    (pl.col("Vol_5d")/pl.col("Vol_20d")).alias("Vol_Ratio"),
    (pl.col("MA_5d")/pl.col("MA_20d")).alias("MA_Ratio"),
    (pl.when((pl.col("High")-pl.col("Low")) !=0)
        .then((pl.col("Close")-pl.col("Low"))/(pl.col("High")-pl.col("Low")))
        .otherwise(0.5)
        .alias("Close_Pos_Range")),
    ((pl.col("Close")-pl.col("Open"))/pl.col("Open")).alias("Intraday_Return"),
    ((pl.col("High")-pl.col("Low"))/pl.col("Close")).rolling_mean(window_size=5).alias("Rolling_Range"),
)

nifty = nifty.drop(["Ret_1d","MA_5d","MA_20d"])

nifty.columns

['Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'Date',
 'Ret_3d',
 'Ret_5d',
 'Ret_20d',
 'Vol_5d',
 'Vol_10d',
 'Vol_20d',
 'Abs_Return',
 'Rolling_Abs_Return',
 'Vol_Ratio',
 'MA_Ratio',
 'Close_Pos_Range',
 'Intraday_Return',
 'Rolling_Range']

In [67]:
gold = gold.with_columns(
    pl.col("Close").pct_change().alias("Ret_1d"),
    pl.col("Close").pct_change(n=3).alias("Ret_3d"),
    pl.col("Close").pct_change(n=5).alias("Ret_5d"),
    pl.col("Close").pct_change(n=20).alias("Ret_20d"),
    pl.col("Close").rolling_mean(window_size=5).alias("MA_5d"),
    pl.col("Close").rolling_mean(window_size=20).alias("MA_20d"),
)
gold = gold.with_columns(
    pl.col("Ret_1d").rolling_std(window_size=5).alias("Vol_5d"),
    pl.col("Ret_1d").rolling_std(window_size=10).alias("Vol_10d"),
    pl.col("Ret_1d").rolling_std(window_size=20).alias("Vol_20d"),
)
gold = gold.with_columns(
    pl.col("Ret_1d").abs().alias("Abs_Return"),
    pl.col("Ret_1d").abs().rolling_mean(window_size=5).alias("Rolling_Abs_Return"),
    (pl.col("Vol_5d")/(pl.col("Vol_20d")+1e-8)).alias("Vol_Ratio"),
    (pl.col("MA_5d")/(pl.col("MA_20d")+1e-8)).alias("MA_Ratio"),
    (pl.when((pl.col("High")-pl.col("Low")) !=0)
        .then((pl.col("Close")-pl.col("Low"))/(pl.col("High")-pl.col("Low")))
        .otherwise(0.5)
        .alias("Close_Pos_Range")),
    ((pl.col("Close")-pl.col("Open"))/pl.col("Open")).alias("Intraday_Return"),
    ((pl.col("High")-pl.col("Low"))/pl.col("Close")).rolling_mean(window_size=5).alias("Rolling_Range"),
)
gold = gold.with_columns(
    (
        (pl.col("Close").shift(-4) - pl.col("Open").shift(-1)) 
        / pl.col("Open").shift(-1)
    ).alias("Forward_Return")
)

gold = gold.with_columns(
    (pl.col("Forward_Return") > 0).cast(pl.Int8).alias("Label")
)

gold = gold.drop(["Ret_1d","MA_5d","MA_20d","Forward_Return"])

gold.columns

['Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'Date',
 'Ret_3d',
 'Ret_5d',
 'Ret_20d',
 'Vol_5d',
 'Vol_10d',
 'Vol_20d',
 'Abs_Return',
 'Rolling_Abs_Return',
 'Vol_Ratio',
 'MA_Ratio',
 'Close_Pos_Range',
 'Intraday_Return',
 'Rolling_Range',
 'Label']

In [68]:
gold.tail()

Close,High,Low,Open,Volume,Date,Ret_3d,Ret_5d,Ret_20d,Vol_5d,Vol_10d,Vol_20d,Abs_Return,Rolling_Abs_Return,Vol_Ratio,MA_Ratio,Close_Pos_Range,Intraday_Return,Rolling_Range,Label
f64,f64,f64,f64,i64,datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
127.419998,132.479996,126.0,129.100006,141941218,2026-01-23 00:00:00,0.012395,0.081848,0.114688,0.056833,0.03927,0.028662,0.024771,0.048681,1.982867,1.082241,0.219136,-0.013013,0.061827,1
131.449997,134.0,129.029999,130.550003,139657546,2026-01-27 00:00:00,-0.02608,0.085467,0.162966,0.05702,0.039459,0.028921,0.031628,0.049369,1.971582,1.091334,0.486921,0.006894,0.062221,null
135.820007,137.100006,132.0,132.0,184193702,2026-01-28 00:00:00,0.092328,0.079136,0.221733,0.056512,0.03951,0.028846,0.033245,0.048157,1.959097,1.096759,0.74902,0.028939,0.059084,null
146.529999,148.139999,141.639999,143.0,271824296,2026-01-29 00:00:00,0.149976,0.085649,0.33076,0.058158,0.044145,0.032197,0.078854,0.049451,1.806301,1.099359,0.752308,0.024685,0.054768,null
131.119995,142.800003,127.099998,138.699997,352204561,2026-01-30 00:00:00,-0.00251,0.054528,0.186392,0.069273,0.05989,0.041922,0.105166,0.054733,1.65244,1.10119,0.256051,-0.05465,0.058062,null


In [72]:
gold["Label"].value_counts()

Label,count
i8,u32
null,4
1,1404
0,1816


In [70]:
usdinr = usdinr.with_columns(
    pl.col("Close").pct_change().alias("Ret_1d"),
    pl.col("Close").pct_change(n=3).alias("Ret_3d"),
    pl.col("Close").pct_change(n=5).alias("Ret_5d"),
    pl.col("Close").pct_change(n=20).alias("Ret_20d"),
    pl.col("Close").rolling_mean(window_size=5).alias("MA_5d"),
    pl.col("Close").rolling_mean(window_size=20).alias("MA_20d"),
)
usdinr = usdinr.with_columns(
    pl.col("Ret_1d").rolling_std(window_size=5).alias("Vol_5d"),
    pl.col("Ret_1d").rolling_std(window_size=10).alias("Vol_10d"),
    pl.col("Ret_1d").rolling_std(window_size=20).alias("Vol_20d"),
)
usdinr = usdinr.with_columns(
    pl.col("Ret_1d").abs().alias("Abs_Return"),
    pl.col("Ret_1d").abs().rolling_mean(window_size=5).alias("Rolling_Abs_Return"),
    (pl.col("Vol_5d")/pl.col("Vol_20d")).alias("Vol_Ratio"),
    (pl.col("MA_5d")/pl.col("MA_20d")).alias("MA_Ratio"),
    (pl.when((pl.col("High")-pl.col("Low")) !=0)
        .then((pl.col("Close")-pl.col("Low"))/(pl.col("High")-pl.col("Low")))
        .otherwise(0.5)
        .alias("Close_Pos_Range")),
    ((pl.col("Close")-pl.col("Open"))/pl.col("Open")).alias("Intraday_Return"),
    ((pl.col("High")-pl.col("Low"))/pl.col("Close")).rolling_mean(window_size=5).alias("Rolling_Range"),
)

usdinr = usdinr.drop(["Ret_1d","MA_5d","MA_20d"])

usdinr.columns

['Close',
 'High',
 'Low',
 'Open',
 'Volume',
 'Date',
 'Ret_3d',
 'Ret_5d',
 'Ret_20d',
 'Vol_5d',
 'Vol_10d',
 'Vol_20d',
 'Abs_Return',
 'Rolling_Abs_Return',
 'Vol_Ratio',
 'MA_Ratio',
 'Close_Pos_Range',
 'Intraday_Return',
 'Rolling_Range']